In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ← Private リポジトリならここに PAT（repo 権限）を入れる

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

# 作業ディレクトリを /content に戻す（再実行時の安全策）
if not Path("train.py").exists() and Path("/content").exists():
    os.chdir("/content")

if not Path("train.py").exists():
    if REPO_DIR.exists() and not (REPO_DIR / "train.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "train.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、セル1で GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("train.py").exists():
        raise FileNotFoundError(
            "clone 失敗。GITHUB_TOKEN を設定するか、ランタイムを再起動して再実行してください。"
        )

# どのルートを通っても必ず最新コードに更新する
print("git pull ...")
!git pull
print("OK:", os.getcwd())
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 学習設定（ここだけ変えれば「すべてのセルを実行」で目的の学習が回る）

**基本フロー: データ生成 → CVAE 学習 → 完了**

- `GEN_*` でパートごとのデータ生成を on/off（既にデータがあれば False でスキップ可）
- `TRAIN_CVAE = True` で、`GEN_*` が True のパートだけ CVAE 学習する


In [ ]:
GEN_BACKING  = True  # バッキング音符データ作成
GEN_LEAD     = True  # リード音符データ作成
GEN_BASS     = True  # ベース音符データ作成
GEN_DRUM     = True  # ドラム音符データ作成

# lead pairs: Colab で作ったら Drive に M-D-HH.zip 保存。次回は最新を展開
FORCE_REBUILD_LEAD_PAIRS = True  # True で MIDI から一式作り直し＋Drive 再アップロード（新 ST 反映時はこれ）
# 節約: 旧 pairs zip を展開 → GAPS 追加 import + OS（ST 再 pair しない）→ Drive 再保存
LEAD_GAPS_BOOST_REUSE = False  # True は旧 ST のまま GAPS だけ足す。FORCE_REBUILD=True のときは無視
LEAD_PAIRS_DRIVE_DIR = "/content/drive/MyDrive/colab_train_lead_pairs"

TRAIN_CVAE   = True  # GEN_* が True のパートだけ CVAE 学習

TRAIN_STRUCTURE_PRIOR = False  # 曲調層: accept + balanced_116（+任意で EMOPIA）
USE_EMOPIA = True             # キー/モード用（BPMなし）。不要なら False

GEN_GUITAR_TECH_APPLY = False  # 奏法付与: EG-Solo+TECHS+依頼ギター
TRAIN_GUITAR_TECH_APPLY = False # note文脈→奏法クラス学習

print(f"GEN_BACKING={GEN_BACKING}  GEN_LEAD={GEN_LEAD}  GEN_BASS={GEN_BASS}  GEN_DRUM={GEN_DRUM}")
print(f"FORCE_REBUILD_LEAD_PAIRS={FORCE_REBUILD_LEAD_PAIRS}  LEAD_GAPS_BOOST_REUSE={LEAD_GAPS_BOOST_REUSE}")
print(f"TRAIN_CVAE={TRAIN_CVAE}")
print(f"TRAIN_STRUCTURE_PRIOR={TRAIN_STRUCTURE_PRIOR}  EMOPIA={USE_EMOPIA}")
print(f"GEN_GUITAR_TECH_APPLY={GEN_GUITAR_TECH_APPLY}  TRAIN_GUITAR_TECH_APPLY={TRAIN_GUITAR_TECH_APPLY}")


## バッキング学習

`GEN_BACKING` が `True` のときのみ以下が実行される（データ作成 → CVAE 学習）。


In [ ]:
if GEN_BACKING:
    SYNTHETIC_COUNT = 8000
    SYNTHETIC_SEED = 42
    SYNTHETIC_BARS = 8  # 8固定（長くするとパッチ爆発でディスク落ち）
    # downbeat_chord | root_per_bar | melody_line | onset_to_full
    PAIR_MODE = "downbeat_chord"

    # 容量不足で落ちた場合: ランタイム初期化後、下を実行してから本セルへ
    # !rm -rf data/pairs data/raw/synthetic

    !python prepare_all.py \
      --synthetic-count {SYNTHETIC_COUNT} \
      --synthetic-seed {SYNTHETIC_SEED} \
      --bars {SYNTHETIC_BARS} \
      --force-regenerate \
      --mode {PAIR_MODE}
else:
    print("GEN_BACKING=False のためスキップ（バッキング用データ作成）")

## リード学習

`GEN_LEAD` 時のみ。実データのみ（SynthTab pick 4589＝mono≥0.3+高音675 + 旧ソロ411 + EGSolo/GuitarSet + GAPS）、合成なし、**一から 40 epoch**。

**pairs の置き場（Colab）**
- 作成後: Drive `LEAD_PAIRS_DRIVE_DIR` に **`M-D-HH.zip`**（例 `11-22-19.zip`。同時間帯は `-MM` 付与）と `latest.txt`（1行目=zip名、続けて `build_seconds` / `build_hms`）
- 展開のみ: `FORCE_REBUILD_LEAD_PAIRS=False` かつ `LEAD_GAPS_BOOST_REUSE=False` → 旧 zip 展開だけ
- **節約（GAPS 40%）**: `FORCE_REBUILD=False` かつ `LEAD_GAPS_BOOST_REUSE=True` → 旧 zip + GAPS top160/OS~40%（**ST は更新されない**）
- 一式作り直し: `FORCE_REBUILD_LEAD_PAIRS=True` → 新 MIDI から ST+GAPS を再 pair（**BOOST より優先**）

学習: FiLMCVAEUNet、lr 1e-3、mask-ratio 0.5、`--no-cache`


In [ ]:
if GEN_LEAD:
    from pathlib import Path
    import shutil
    import zipfile
    import subprocess
    from datetime import datetime

    try:
        from zoneinfo import ZoneInfo
        _now = datetime.now(ZoneInfo('Asia/Tokyo'))
    except Exception:
        _now = datetime.now()

    PAIRS_DIR = Path('data/pairs/lead')
    PICK_ZIP = Path('lead_data/synthtab_pick5000.zip')
    LEGACY_ZIP = Path('lead_data/synthtab_lead_merged.zip')
    GAPS_ZIP = Path('lead_data/gaps_v1_midi.zip')
    OUT_DIR = Path('/content/synthtab_pick5000')
    GAPS_DIR = Path('/content/gaps_midi')
    DRIVE_DIR = Path(LEAD_PAIRS_DRIVE_DIR)

    def _count_tonal(root: Path) -> int:
        if not root.is_dir():
            return 0
        return sum(1 for _ in root.rglob('*_tonal.npy'))

    def _mount_drive() -> None:
        try:
            from google.colab import drive
        except ImportError:
            print('Drive mount スキップ（Colab 外）')
            return
        mount = Path('/content/drive')
        if not (mount / 'MyDrive').exists():
            drive.mount('/content/drive')

    def _dated_zip_name(dt: datetime) -> str:
        # 月-日-時.zip / 衝突時 月-日-時-分.zip
        base = f'{dt.month}-{dt.day}-{dt.hour:02d}'
        name = f'{base}.zip'
        if (DRIVE_DIR / name).exists():
            name = f'{base}-{dt.minute:02d}.zip'
        return name

    def _pick_drive_zip() -> Path | None:
        if not DRIVE_DIR.is_dir():
            return None
        latest = DRIVE_DIR / 'latest.txt'
        if latest.is_file():
            # 1行目が zip 名。以降に所要時間などがあっても無視
            name = latest.read_text(encoding='utf-8').strip().splitlines()[0].strip()
            cand = DRIVE_DIR / name
            if cand.is_file():
                return cand
        zips = sorted(
            DRIVE_DIR.glob('*.zip'),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        return zips[0] if zips else None

    def _extract_pairs_zip(zpath: Path) -> None:
        if PAIRS_DIR.exists():
            shutil.rmtree(PAIRS_DIR)
        PAIRS_DIR.mkdir(parents=True, exist_ok=True)
        print(f'展開中: {zpath} ...')
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(PAIRS_DIR)
        # zip が lead/ を含む場合は中身を上げる
        nested = PAIRS_DIR / 'lead'
        if nested.is_dir() and not (PAIRS_DIR / 'target').exists():
            for child in list(nested.iterdir()):
                dest = PAIRS_DIR / child.name
                if dest.exists():
                    shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
                shutil.move(str(child), str(dest))
            shutil.rmtree(nested)
        n = _count_tonal(PAIRS_DIR / 'target')
        print(f'lead pairs ready: {n} tonal ← {zpath.name}')
        if n <= 0:
            raise RuntimeError(f'zip に tonal がありません: {zpath}')

    def _build_from_midi() -> None:
        if not PICK_ZIP.is_file():
            raise FileNotFoundError(f'{PICK_ZIP} がありません')
        stamp = OUT_DIR / '.zip_mtime'
        mtime = str(PICK_ZIP.stat().st_mtime_ns)
        n_mid = sum(1 for _ in OUT_DIR.rglob('*.mid')) if OUT_DIR.exists() else 0
        if not (stamp.is_file() and stamp.read_text(encoding='utf-8').strip() == mtime and n_mid > 0):
            if OUT_DIR.exists():
                shutil.rmtree(OUT_DIR)
            OUT_DIR.mkdir(parents=True, exist_ok=True)
            print(f'展開中: {PICK_ZIP.name} ...')
            with zipfile.ZipFile(PICK_ZIP) as zf:
                zf.extractall(OUT_DIR)
            stamp.write_text(mtime, encoding='utf-8')
        EXTRA_SOLO = Path('lead_data/synthtab_extra_solo411.txt')
        extra_names = set()
        if EXTRA_SOLO.is_file():
            extra_names = {
                ln.strip() for ln in EXTRA_SOLO.read_text(encoding='utf-8').splitlines() if ln.strip()
            }
        if LEGACY_ZIP.is_file():
            with zipfile.ZipFile(LEGACY_ZIP) as zf:
                for name in zf.namelist():
                    base = Path(name).name
                    want = (
                        base.startswith(('EGSolo', 'GuitarSet'))
                        or base in extra_names
                    )
                    if want and base.lower().endswith('.mid'):
                        dest = OUT_DIR / base
                        if not dest.exists():
                            dest.write_bytes(zf.read(name))
        if not GAPS_ZIP.is_file():
            raise FileNotFoundError(f'{GAPS_ZIP} がありません')
        g_stamp = GAPS_DIR / '.zip_mtime'
        g_mtime = str(GAPS_ZIP.stat().st_mtime_ns)
        g_n = sum(1 for _ in GAPS_DIR.rglob('*.mid')) if GAPS_DIR.exists() else 0
        if not (g_stamp.is_file() and g_stamp.read_text(encoding='utf-8').strip() == g_mtime and g_n > 0):
            if GAPS_DIR.exists():
                shutil.rmtree(GAPS_DIR)
            GAPS_DIR.mkdir(parents=True, exist_ok=True)
            print(f'展開中: {GAPS_ZIP.name} ...')
            with zipfile.ZipFile(GAPS_ZIP) as zf:
                zf.extractall(GAPS_DIR)
            g_stamp.write_text(g_mtime, encoding='utf-8')
        print(f'lead mid: {sum(1 for _ in OUT_DIR.rglob("*.mid"))}  gaps mid: {sum(1 for _ in GAPS_DIR.rglob("*.mid"))}')
        cmd = [
            'python', 'build_lead_pairs_mixed.py',
            '--pairs-dir', str(PAIRS_DIR),
            '--clear',
            '--skip-synthetic',
            '--synthtab-dir', str(OUT_DIR),
            '--synthtab-min-bars', '8',
            '--synthtab-no-keyword-filter',
            '--gaps-dir', str(GAPS_DIR),
            '--gaps-top-k', '160',
            '--gaps-target-ratio', '0.40',
            '--gaps-min-ratio', '0.38',
            '--gaps-max-ratio', '0.45',
        ]
        print('+', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    def _fmt_elapsed(sec: float) -> str:
        sec_i = int(round(sec))
        h, rem = divmod(sec_i, 3600)
        m, s = divmod(rem, 60)
        if h:
            return f'{h}h{m:02d}m{s:02d}s'
        return f'{m}m{s:02d}s'

    def _upload_pairs_to_drive(*, build_seconds: float | None = None) -> Path:
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        zname = _dated_zip_name(_now)
        zpath = DRIVE_DIR / zname
        tmp = Path('/tmp') / zname
        if tmp.exists():
            tmp.unlink()
        print(f'Drive 用 zip 作成中 → {zpath} ...')
        with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            for p in PAIRS_DIR.rglob('*'):
                if p.is_file() and p.name != '.zip_mtime':
                    zf.write(p, arcname=str(p.relative_to(PAIRS_DIR)))
        shutil.copy2(tmp, zpath)
        lines = [zname]
        if build_seconds is not None:
            lines.append(f'build_seconds={build_seconds:.1f}')
            lines.append(f'build_hms={_fmt_elapsed(build_seconds)}')
        (DRIVE_DIR / 'latest.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
        extra = f'  build={_fmt_elapsed(build_seconds)}' if build_seconds is not None else ''
        print(f'Drive 保存: {zpath}  latest.txt={zname}{extra}  tonal={_count_tonal(PAIRS_DIR / "target")}')
        return zpath

    def _ensure_gaps_midi() -> None:
        if not GAPS_ZIP.is_file():
            raise FileNotFoundError(f'{GAPS_ZIP} がありません')
        g_stamp = GAPS_DIR / '.zip_mtime'
        g_mtime = str(GAPS_ZIP.stat().st_mtime_ns)
        g_n = sum(1 for _ in GAPS_DIR.rglob('*.mid')) if GAPS_DIR.exists() else 0
        if not (g_stamp.is_file() and g_stamp.read_text(encoding='utf-8').strip() == g_mtime and g_n > 0):
            if GAPS_DIR.exists():
                shutil.rmtree(GAPS_DIR)
            GAPS_DIR.mkdir(parents=True, exist_ok=True)
            print(f'展開中: {GAPS_ZIP.name} ...')
            with zipfile.ZipFile(GAPS_ZIP) as zf:
                zf.extractall(GAPS_DIR)
            g_stamp.write_text(g_mtime, encoding='utf-8')
        print(f'gaps mid: {sum(1 for _ in GAPS_DIR.rglob("*.mid"))}')

    def _boost_gaps_on_existing_pairs() -> None:
        """旧 pairs を残したまま GAPS top-k / OS だけ更新（ST 再 pair しない）。"""
        import time
        z_existing = _pick_drive_zip()
        if z_existing is None:
            raise FileNotFoundError(
                f'Drive に lead pairs zip がありません（節約不可）: {DRIVE_DIR}'
            )
        t0 = time.perf_counter()
        _extract_pairs_zip(z_existing)
        before = _count_tonal(PAIRS_DIR / 'target')
        print(f'展開後 tonal={before}')
        _ensure_gaps_midi()
        cmd = [
            'python', 'build_lead_pairs_mixed.py',
            '--pairs-dir', str(PAIRS_DIR),
            # --clear しない（ST / 既存を保持）
            '--skip-synthtab',
            '--skip-animetab',
            '--skip-synthetic',
            '--gaps-dir', str(GAPS_DIR),
            '--gaps-top-k', '160',
            '--gaps-target-ratio', '0.40',
            '--gaps-min-ratio', '0.38',
            '--gaps-max-ratio', '0.45',
        ]
        print('実行:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        after = _count_tonal(PAIRS_DIR / 'target')
        build_sec = time.perf_counter() - t0
        print(f'GAPS boost 所要: {_fmt_elapsed(build_sec)}  tonal {before} → {after}')
        _upload_pairs_to_drive(build_seconds=build_sec)

    _mount_drive()
    if FORCE_REBUILD_LEAD_PAIRS:
        if LEAD_GAPS_BOOST_REUSE:
            print('LEAD_GAPS_BOOST_REUSE は無視（FORCE_REBUILD_LEAD_PAIRS=True が優先）')
        print('FORCE_REBUILD_LEAD_PAIRS=True → MIDI から一式 pair 化（新 ST + GAPS）')
        import time
        t0 = time.perf_counter()
        _build_from_midi()
        build_sec = time.perf_counter() - t0
        print(f'pair 化所要: {_fmt_elapsed(build_sec)} ({build_sec:.1f}s)')
        _upload_pairs_to_drive(build_seconds=build_sec)
    elif LEAD_GAPS_BOOST_REUSE:
        print('LEAD_GAPS_BOOST_REUSE=True → 旧 pairs + GAPS 40% OS（ST 再 pair なし）')
        _boost_gaps_on_existing_pairs()
    else:
        z_existing = _pick_drive_zip()
        if z_existing is not None:
            _extract_pairs_zip(z_existing)
        else:
            print(f'Drive に lead pairs zip なし: {DRIVE_DIR}')
            print('Colab 上で pair 化します（完了後 Drive に M-D-HH.zip 保存）')
            import time
            t0 = time.perf_counter()
            _build_from_midi()
            build_sec = time.perf_counter() - t0
            print(f'pair 化所要: {_fmt_elapsed(build_sec)} ({build_sec:.1f}s)')
            _upload_pairs_to_drive(build_seconds=build_sec)
else:
    print('GEN_LEAD=False のためスキップ')


## ベース学習

`GEN_BASS` が `True` のときのみ実行。入力はギターコード骨格、正解はベースライン（category 4）。
ドラムのキック位置と揃えた合成データを使う。checkpoint は `checkpoints/bass`。
- 入力: コード骨格 11ch + BPM 1ch（計 12ch）
- 正解: tonal 11ch（cat4 のみ非ゼロ）
- 学習: `train.py` で **40 epoch**（FiLMCVAEUNet、mask-ratio 0.3）


In [ ]:
if GEN_BASS:
    BASS_COUNT = 2000
    !python generate_bass_pairs.py \
      --pairs-dir data/pairs/bass \
      --count {BASS_COUNT} \
      --seed 42
else:
    print("GEN_BASS=False のためスキップ（ベース用データ作成）")


## ドラム学習（beat_type 条件・in=24）

`GEN_DRUM` が `True` のときのみ実行。入力はギターコード骨格、正解は GM ドラムパターン（1ch 二値）。
`train_drum.py`（BCEWithLogitsLoss）で学習。checkpoint は `checkpoints/drum`。

- 入力: コード骨格 11ch + BPM 1ch + beat_type one-hot 12ch（**計 24ch**）
- 正解: `*_drum.npy` (1,128,128) 0/1。条件: `*_beat.npy`（型 ID）
- データ: **7200 本**（12型 × 600）。旧データは `*_beat.npy` なしのため再生成必須
- 学習: **40 epoch**、pos_weight=20。`build_unet` が in_ch を自動検出


In [ ]:
if GEN_DRUM:
    DRUM_COUNT = 7200  # 12型 × 600。旧データは *_beat.npy なしのため再生成必須
    !python generate_drum_pairs.py \
      --pairs-dir data/pairs/drum \
      --count {DRUM_COUNT} \
      --seed 42
else:
    print("GEN_DRUM=False のためスキップ（ドラム用データ作成）")


## CVAE 学習（GEN_* が True のパートだけ）

`TRAIN_CVAE` が `True` のとき、`GEN_*` が `True` のモデルだけ CVAE 学習する。

- **FiLMCVAEUNet**: backing / lead / bass に適用。z を全デコーダ層に FiLM で注入し posterior collapse を防ぐ
- **CVAEUNet（旧）**: drum のみ。スキップ接続経由で z を入力に concat する従来方式
- 学習時に `z ~ q(z|x,y)` からサンプリングし、再構成 loss + KL loss で学習
- 推論時は `z ~ N(0,I)` → 同じ入力でも毎回異なる出力を生成
- ckpt に `model_type: "cvae_film"` が保存され、`load_model` が自動判別する
- 既存 ckpt を上書きするので必要ならバックアップを取ること

共通: `--beta 0.01 --free-bits 0.5`。mask-ratio は 0.5(lead/backing) / 0.3(bass)。
epoch と KL anneal はパート別（backing 40 + onset/midbar + **require-groove-cond** in=20 / lead 40 / bass 20 / drum 30）。

backing 入力: tonal11 + BPM1 + groove one-hot8（7 stroke + arp）。旧 12ch pairs は不可 → `prepare_all --force-regenerate`（8000）。

**AMP**: `train.py` 既定は **OFF（fp32）**。AdaIN/FiLM + AMP だと重みが NaN 化し `backing=0` になる。`--amp` は付けない。NaN の ckpt から `--resume` しても直らない（新規学習）。

**Dataset cache**: 既定 **OFF**（`--no-cache`）。数万 pairs の起動時全件 RAM 展開は Colab が途中 Interrupt しやすい。必要なら `--cache-dataset`。

**傾向確認**: 該当の `CVAE_EPOCHS_*` を 10 にする。ログの `kl` が `free_bits（0.5）` 付近なら崩壊していない。


In [ ]:
if TRAIN_CVAE:
    # backing: ストローク空きが smear するので epoch↑ + onset/中拍を損失で押す（データ作り直しではない）
    CVAE_EPOCHS_BACKING = 40
    CVAE_EPOCHS_LEAD    = 40
    CVAE_EPOCHS_BASS    = 20
    CVAE_EPOCHS_DRUM    = 30
    CVAE_BATCH_SIZE = 32
    CVAE_BETA = 0.01
    CVAE_LATENT_DIM = 16
    CVAE_FREE_BITS = 0.5      # z の各次元が持つべき最低情報量。これ以下に KL を下げられない → z 無視を防ぐ
    CVAE_MASK_RATIO_LEAD    = 0.5  # lead: 骨格を半分隠して z にフレーズを覚えさせる
    CVAE_MASK_RATIO_BACKING = 0.5  # backing: 同上
    CVAE_MASK_RATIO_BASS    = 0.3  # bass: 弱め（ルート音は予測しやすい）
    CVAE_ONSET_WEIGHT_BACKING = 4.0       # onset セルを強く合わせる
    CVAE_MIDBAR_ONSET_BACKING = 2.0       # 小節頭以外の打鍵を強化
    ran = False

    if GEN_BACKING:
        ran = True
        print("=== backing CVAE ===")
        !python train.py \
          --pairs-dir data/pairs/synthetic \
          --epochs {CVAE_EPOCHS_BACKING} \
          --batch-size {CVAE_BATCH_SIZE} \
          --lr 1e-3 \
          --checkpoint-dir checkpoints/backing \
          --pos-weight 20 \
          --onset-weight {CVAE_ONSET_WEIGHT_BACKING} \
          --midbar-onset-bonus {CVAE_MIDBAR_ONSET_BACKING} \
          --require-groove-cond \
          --cvae \
          --use-film \
          --latent-dim {CVAE_LATENT_DIM} \
          --beta {CVAE_BETA} \
          --kl-anneal-epochs {CVAE_EPOCHS_BACKING} \
          --free-bits {CVAE_FREE_BITS} \
          --mask-ratio {CVAE_MASK_RATIO_BACKING}
    else:
        print("GEN_BACKING=False のため backing CVAE はスキップ")

    if GEN_LEAD:
        ran = True
        print("=== lead CVAE（一から）===")
        !python train.py \
          --pairs-dir data/pairs/lead \
          --epochs {CVAE_EPOCHS_LEAD} \
          --batch-size {CVAE_BATCH_SIZE} \
          --lr 1e-3 \
          --checkpoint-dir checkpoints/lead \
          --pos-weight 20 \
          --require-power-cond \
          --cvae \
          --use-film \
          --latent-dim {CVAE_LATENT_DIM} \
          --beta {CVAE_BETA} \
          --kl-anneal-epochs {CVAE_EPOCHS_LEAD} \
          --free-bits {CVAE_FREE_BITS} \
          --mask-ratio {CVAE_MASK_RATIO_LEAD} \
          --no-cache
    else:
        print("GEN_LEAD=False のため lead CVAE はスキップ")

    if GEN_BASS:
        ran = True
        print("=== bass CVAE ===")
        !python train.py \
          --pairs-dir data/pairs/bass \
          --epochs {CVAE_EPOCHS_BASS} \
          --batch-size {CVAE_BATCH_SIZE} \
          --lr 1e-3 \
          --checkpoint-dir checkpoints/bass \
          --pos-weight 20 \
          --cvae \
          --use-film \
          --latent-dim {CVAE_LATENT_DIM} \
          --beta {CVAE_BETA} \
          --kl-anneal-epochs {CVAE_EPOCHS_BASS} \
          --free-bits {CVAE_FREE_BITS} \
          --mask-ratio {CVAE_MASK_RATIO_BASS}
    else:
        print("GEN_BASS=False のため bass CVAE はスキップ")

    if GEN_DRUM:
        ran = True
        print("=== drum CVAE ===")
        !python train_drum.py \
          --pairs-dir data/pairs/drum \
          --epochs {CVAE_EPOCHS_DRUM} \
          --batch-size {CVAE_BATCH_SIZE} \
          --lr 1e-3 \
          --checkpoint-dir checkpoints/drum \
          --pos-weight 20 \
          --cvae \
          --latent-dim {CVAE_LATENT_DIM} \
          --beta {CVAE_BETA} \
          --kl-anneal-epochs {CVAE_EPOCHS_DRUM}
    else:
        print("GEN_DRUM=False のため drum CVAE はスキップ")

    if not ran:
        print("TRAIN_CVAE=True だが GEN_* がすべて False のため、CVAE は学習しません")
else:
    print("TRAIN_CVAE=False のためスキップ（CVAE 学習）")


## 曲調層 Structure Prior 学習（VA → BPM / キー / モード / 進行）

`TRAIN_STRUCTURE_PRIOR` が True のとき実行。入力は VA 2次元。BPM はビン分布サンプリング。

| データ | 役割 |
|---|---|
| `accept.jsonl` | 進行・キー・BPM（`--jsonl`） |
| `structure_pairs_balanced_116.jsonl` | VA帯×116×4（VA→BPM 本線 extras） |
| EMOPIA（`USE_EMOPIA`） | キー／モードのみ（任意） |

VGMIDI / PMEmo / FME 全量や high_bpm は extras に載せない（4×116 の揃えを崩す）。


In [ ]:
from pathlib import Path

if TRAIN_STRUCTURE_PRIOR:
    print("=== VA×BPM organize (balanced_116) ===")
    !python scripts/organize_va_bpm_pairs.py --out-dir data/va_bpm_curated --per-group 116 --seed 0
    PRIOR_EPOCHS = 80
    PRIOR_BATCH_SIZE = 32
    extras = []

    # VA→BPM 本線: 4×116 のみ（VGMIDI/PMEmo/FME 全量は載せない＝件数揃えを崩さない）
    bal = Path("data/va_bpm_curated/structure_pairs_balanced_116.jsonl")
    if bal.is_file():
        print(f"=== balanced_116: {bal} ===")
        extras.append(str(bal))
    else:
        raise FileNotFoundError(
            "structure_pairs_balanced_116.jsonl がありません。"
            " data/va_bpm_curated/ を pull するか organize を実行してください。"
        )

    if USE_EMOPIA:
        em_out = Path("data/emopia_pairs/structure_pairs.jsonl")
        if em_out.is_file():
            print(f"=== EMOPIA (key/mode): {em_out} ===")
            extras.append("data/emopia_pairs/structure_pairs.jsonl")
        else:
            print("=== EMOPIA pairs missing; skip ===")

    EXTRA = ("--extra-jsonl " + " ".join(extras)) if extras else ""
    print("extra_jsonl:", extras)

    print("=== structure prior 学習 ===")
    !python scripts/train_structure_prior.py \
      --jsonl data/prior_pairs/manifests/accept.jsonl \
      {EXTRA} \
      --checkpoint-dir checkpoints/structure_prior \
      --epochs {PRIOR_EPOCHS} \
      --batch-size {PRIOR_BATCH_SIZE}
else:
    print("TRAIN_STRUCTURE_PRIOR=False のためスキップ（Structure Prior）")


## ギター奏法付与（TechApply）学習

`GEN_GUITAR_TECH_APPLY` / `TRAIN_GUITAR_TECH_APPLY` が True のとき実行。

| 段階 | 内容 |
|---|---|
| データ | EG-Solo + Guitar-TECHS + 依頼ギター（labels JSON） |
| 学習 | note 文脈特徴 → 奏法クラス（`TechApplyNet`） |
| 出力 | `checkpoints/guitar_tech/guitar_tech_apply_last.pt` |

ローカル配置: `prttype/checkpoints/guitar_tech/guitar_tech_apply_last.pt`（生成時 `--apply-tech`）。


In [ ]:
from pathlib import Path

if GEN_GUITAR_TECH_APPLY:
    print("=== guitar tech apply dataset (EG-Solo+TECHS+friend) ===")
    !python -m guitar_tech.build_apply_dataset \
      --out-dir data/ギター_dataset/apply
else:
    print("GEN_GUITAR_TECH_APPLY=False のためスキップ（奏法 apply データ）")

if TRAIN_GUITAR_TECH_APPLY:
    APPLY_EPOCHS = 40
    APPLY_BATCH = 256
    print("=== guitar tech apply 学習 ===")
    !python -m guitar_tech.train_apply \
      --dataset-dir data/ギター_dataset/apply \
      --checkpoint-dir checkpoints/guitar_tech \
      --epochs {APPLY_EPOCHS} \
      --batch-size {APPLY_BATCH}
    ckpt = Path("checkpoints/guitar_tech/guitar_tech_apply_last.pt")
    if not ckpt.is_file():
        raise FileNotFoundError(ckpt)
    print(f"ckpt: {ckpt}")
else:
    print("TRAIN_GUITAR_TECH_APPLY=False のためスキップ（奏法 apply 学習）")


## 実行サマリ

In [ ]:
from pathlib import Path

print("=== 学習設定 ===")
print(f"GEN_BACKING={GEN_BACKING}  GEN_LEAD={GEN_LEAD}  GEN_BASS={GEN_BASS}  GEN_DRUM={GEN_DRUM}")
print(f"TRAIN_CVAE={TRAIN_CVAE}  TRAIN_STRUCTURE_PRIOR={TRAIN_STRUCTURE_PRIOR}  EMOPIA={USE_EMOPIA}")
print(f"GEN_GUITAR_TECH_APPLY={GEN_GUITAR_TECH_APPLY}  TRAIN_GUITAR_TECH_APPLY={TRAIN_GUITAR_TECH_APPLY}")

print("\n=== checkpoints ===")
for d in [
    "checkpoints/stage1",
    "checkpoints/lead",
    "checkpoints/bass",
    "checkpoints/drum",
    "checkpoints/structure_prior",
    "checkpoints/guitar_tech",
]:
    pt = list(Path(d).glob("*.pt")) if Path(d).is_dir() else []
    print(f"  {d}: {[x.name for x in pt] if pt else '(なし)'}")


## Drive に ckpt 保存

学習後に `*_cvae_last.pt` だけを指定フォルダへアップロードする（同名があれば上書き）。
初回は Google 認証が出る。

In [ ]:
SAVE_CKPT_TO_DRIVE = True
DRIVE_FOLDER_ID = "1cwMHlT2o_D2m09CSJAGH_W784OJdKUzh"

if SAVE_CKPT_TO_DRIVE:
    from pathlib import Path

    from google.colab import auth
    from google.auth import default
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    auth.authenticate_user()
    creds, _ = default()
    service = build("drive", "v3", credentials=creds)

    ckpts = sorted(Path("checkpoints").rglob("*_cvae_last.pt"))
    if not ckpts:
        print("*_cvae_last.pt がありません")
    else:
        for ckpt in ckpts:
            q = (
                f"name='{ckpt.name}' and '{DRIVE_FOLDER_ID}' in parents "
                "and trashed=false"
            )
            existing = (
                service.files()
                .list(
                    q=q,
                    spaces="drive",
                    fields="files(id, name)",
                    supportsAllDrives=True,
                    includeItemsFromAllDrives=True,
                )
                .execute()
                .get("files", [])
            )
            media = MediaFileUpload(str(ckpt), resumable=True)
            if existing:
                service.files().update(
                    fileId=existing[0]["id"],
                    media_body=media,
                    supportsAllDrives=True,
                ).execute()
                print(f"updated: {ckpt}")
            else:
                service.files().create(
                    body={"name": ckpt.name, "parents": [DRIVE_FOLDER_ID]},
                    media_body=media,
                    fields="id",
                    supportsAllDrives=True,
                ).execute()
                print(f"uploaded: {ckpt}")
else:
    print("SAVE_CKPT_TO_DRIVE=False のためスキップ")

## （確認用）SongFormer × SynthTab スパイク

区間ラベルが SynthTab MIDI レンダで取れるかの確認だけ。本学習パイプラインには未接続。
`SPIKE_SONGFORMER = True` のとき数曲推論 → `/content/sf_spike.json`。
Python 依存は上の `requirements.txt`。レンダ用に fluidsynth だけ apt。GPU 前提。

In [ ]:
SPIKE_SONGFORMER = False  # 確認するときだけ True
SPIKE_N = 5

if SPIKE_SONGFORMER:
    from pathlib import Path
    import zipfile

    SYNTHTAB_ZIP = Path('lead_data/synthtab_lead_merged.zip')
    SYNTHTAB_DIR = Path('/content/synthtab_lead')
    if not SYNTHTAB_DIR.is_dir() or not any(SYNTHTAB_DIR.rglob('*.mid')):
        if not SYNTHTAB_ZIP.is_file():
            raise FileNotFoundError(SYNTHTAB_ZIP)
        SYNTHTAB_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SYNTHTAB_ZIP) as zf:
            zf.extractall(SYNTHTAB_DIR)

    # OS パッケージのみ（pip 依存は requirements.txt）
    !apt-get -qq install -y fluidsynth fluid-soundfont-gm

    import torch
    from lead_data.label_synthtab_sections import spike_songformer

    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    print('device:', device)
    spike_songformer(
        SYNTHTAB_DIR,
        n=SPIKE_N,
        device=device,
        out_json=Path('/content/sf_spike.json'),
        min_bars=8,
    )
    print('結果: /content/sf_spike.json')
else:
    print('SPIKE_SONGFORMER=False のためスキップ（SongFormer 確認）')